# 📓 Llama-Index Practice

In this section, we will create a simple Llama Index app and learn how to log and get feedback on an LLM response. Additionally, we will explore how the query engine can be modified based on its intended purpose.​

Through this process, we hope to gain a clear understanding of the role and functionalities of the Llama Index, as well as the specific inputs and outputs of each method.  

<img src="https://miro.medium.com/v2/resize:fit:1400/0*CrywD0tloiK9dgy_.png" width="600">

This section is divided into three main stages:

###I. Build a Query Engine  
###II. Use Database Management Method  
###III. Make a Custom Query Engine

Let's first look at what is needed to build a Query Engine.

## I. Buliding a Query Engine

The task of this section is to create a Query Engine that takes a user's query, retrieves for related content, and returns a final summary.  

Ignore the `Vector DB` in the picture. We don't use external vector database in the pracetice. Instead, we will use local memory as simple vector store.


<img src="https://i.imgur.com/bR4xaBd.png">

To accomplish this task, the following steps will be taken.
<br/>

#### 1. Install and Import Libraries
#### 2. DownLoad Data
#### 3. Make Query Engine from Index Directly
#### 4. Conduct Activity


### 1. Install and Import Libraries

In [1]:
### YOUR CODE HERE ###
import os
os.environ.pop("PYTHONPATH", None)

! pip install llama_index==0.12.2 openai==1.55.3 packaging>=24.0 streamlit==1.35.0 --quiet

In [1]:
### YOUR CODE HERE ###

import os
os.environ["OPENAI_API_KEY"] = 

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document, PromptTemplate
from openai import OpenAI

from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer

import textwrap
import openai

### 2. DownLoad Data

Previously, we learned that the Query Engine receives the following elements as input and output.
<br/>
#### Input: **User Query**, **Vector Database**
#### Output: **Summary of Retrieved Passages**
<br/>

Here, the User Query is an element that we can decide. However, since the Vector Database must contain information related to the Query, we need to consider how to configure the Vector Database before proceeding with the practice.

The Vector Database can be divided into two categories: a storage that allows the query engine to retrieve related passages, and the data containing the related passages. In this practice, we will use local memory as a data storage instead of an external Vector Database, so the Vector Database will be referred to as a Vector Store from now on. For the data, we will use a simple txt file provided by Llama Index.

Before we begin the practical exercises, let's first take a look at the txt file we used. The code below can be used to download the txt file we used with the `wget` command.


In [2]:
### YOUR CODE HERE ###

import os, urllib.request

os.makedirs("data", exist_ok=True)
url = "https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt"
urllib.request.urlretrieve(url, "data/paul_graham_essay.txt")

('data/paul_graham_essay.txt', <http.client.HTTPMessage at 0x1552172e9f0>)

If there are no issues with the download, you should be able to see that the `paul_graham_essay.txt` file has been downloaded.

This example uses the text of Paul Graham’s essay, [“What I Worked On”](https://paulgraham.com/worked.html), and is the canonical llama-index example.


Let’s print the first fifteen lines of the txt file. This will allow us to see that the text file deals with the experiences and reflections of one person's life.


In [3]:
import os
print(os.getcwd())

C:\ai-specialist\RAG\실습자료\1일차\Code


In [4]:
### YOUR CODE HERE ###

path_to_txt = r'./data/paul_graham_essay.txt' #change this path

with open(path_to_txt, 'r', encoding='utf-8') as file:
    for i in range(15):
        line = file.readline()
        if not line:
            break
        print(line.strip())
print("\n...")



What I Worked On

February 2021

Before college the two main things I worked on, outside of school, were writing and programming. I didn't write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.

The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district's 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.

The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in t

### 3. Make Query Engine from Index Directly

Now let's create a query engine unsing the above information.

First, since Llama Index cannot directly handle text files in their raw form, we must convert them into Documents and Nodes. This can be accomplished with the following code.

```Python
documents = SimpleDirectoryReader("data").load_data()
```

SimpleDirectoryReader takes the directory path containing data that will go into the Vector Store as input and converts all the files within that directory into Document objects. The data types that can be converted into Documents are as follows.


*   .txt - text file  
*   .csv - comma-separated values  
*   .docx - Microsoft Word  
*   .epub - EPUB ebook format  
*   .hwp - Hangul Word Processor  
*   .ipynb - Jupyter Notebook  
*   .jpeg, .jpg - JPEG image  
*   .mbox - MBOX email archive  
*   .md - Markdown  
*   .mp3, .mp4 - audio and video  
*   .pdf - Portable Document Format  
*   .png - Portable Network Graphics  
*   .ppt, .pptm, .pptx - Microsoft PowerPoint  

You can view the text contained in the `documents` using the following command:

```Python
for i in range(len(documents)):
  print(documents[i].text)
```

In [5]:
### YOUR CODE HERE ###

documents = SimpleDirectoryReader("data").load_data()
documents

[Document(id_='2378c500-c68d-4a53-aee3-539837b0d113', embedding=None, metadata={'file_path': 'C:\\ai-specialist\\RAG\\실습자료\\1일차\\Code\\data\\paul_graham_essay.txt', 'file_name': 'paul_graham_essay.txt', 'file_type': 'text/plain', 'file_size': 75042, 'creation_date': '2026-10-22', 'last_modified_date': '2026-10-22'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='\n\nWhat I Worked On\n\nFebruary 2021\n\nBefore college the two main things I worked on, outside of school, were writing and programming. I didn\'t write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awfu

In [7]:
from llama_index.core.node_parser import SentenceSplitter
index = VectorStoreIndex.from_documents(documents)

In [19]:
splitter = SentenceSplitter(chunk_size = 512, chunk_overlap = 200)
index = VectorStoreIndex.from_documents(documents, transformations=[splitter])
index.

SyntaxError: invalid syntax (1061207473.py, line 3)

{'1c6d8237-f201-4aa5-8119-ae4b16a5b0ca': '1c6d8237-f201-4aa5-8119-ae4b16a5b0ca',
 '80505bd7-ece1-47c6-99dc-f97a665b17f8': '80505bd7-ece1-47c6-99dc-f97a665b17f8',
 '5d1402c1-adb2-41c8-9834-803088961e31': '5d1402c1-adb2-41c8-9834-803088961e31',
 '707801ea-8c16-4d42-a730-b0fba80ee387': '707801ea-8c16-4d42-a730-b0fba80ee387',
 '7f797003-4a3e-4fde-b1d7-c330ee441470': '7f797003-4a3e-4fde-b1d7-c330ee441470',
 '6fd50774-a19c-4f51-92ce-1543fb98242f': '6fd50774-a19c-4f51-92ce-1543fb98242f',
 '7938ce08-3654-4d3f-852f-3cee5df4d0d0': '7938ce08-3654-4d3f-852f-3cee5df4d0d0',
 '324d07eb-631f-4254-bec1-bdae6c7c6070': '324d07eb-631f-4254-bec1-bdae6c7c6070',
 'd0d04974-3100-41ff-b647-00a44546f340': 'd0d04974-3100-41ff-b647-00a44546f340',
 '3683b94e-0f6c-4b84-9614-0859c54b3e19': '3683b94e-0f6c-4b84-9614-0859c54b3e19',
 '8e80aad6-c67b-4b90-9827-6e978bada996': '8e80aad6-c67b-4b90-9827-6e978bada996',
 '5827d5ac-16a3-43a7-8eb7-64691236a3ac': '5827d5ac-16a3-43a7-8eb7-64691236a3ac',
 '4569ba85-2ba3-4dfb-85c7-35

In [7]:
### YOUR CODE HERE ###

for i in range(len(documents)):
  print(documents[i].text[:53])
  break



What I Worked On

February 2021

Before college the


Once the Document is complete, we need to use this data to create an index.

The index is created using the VectorStoreIndex.from_documents() method. This method takes a list of Node objects or a Document as input.


```Python
index = VectorStoreIndex.from_documents(documents)
```

If connected to an external vector database, Llama Index follows the index algorithm of that vector database. If there is no connection to an external vector database, it stores the data in the local memory in the form of a dictionary.

In [8]:
### YOUR CODE HERE ###

index = VectorStoreIndex.from_documents(documents)

Now, let’s look at how the index is structured.  

Our index is built based on documents. However, as mentioned in the previous presentation, when Llamaindex stores data in the database, it breaks down the document into nodes. Therefore, let’s first check if the document has been divided into nodes.

In [18]:
### YOUR CODE HERE ###

node_id = index.index_struct.nodes_dict

for key, value in node_id.items():
    print(f"key: {key}, value: {value}")
    node_example_id = value
    break

print("The number of nodes: ", len(node_id.values()))

print(node_id.values())

key: f7319b9b-c485-4b4b-bd6c-6e04758e0aae, value: f7319b9b-c485-4b4b-bd6c-6e04758e0aae
The number of nodes:  22
dict_values(['f7319b9b-c485-4b4b-bd6c-6e04758e0aae', '4fa34dc6-b784-4433-aab7-1e6b7dc2be25', 'd5192e55-ae53-4fb9-8535-1e6cbbc8da4f', '693e68bd-c44a-4ca1-b2b1-df39e793ecae', '61be0c56-f9bf-40d5-a0f5-9fb6278beae2', 'e4b6ab80-5241-4673-a5f3-25a19dfb339b', '24b8c1d9-7e3a-482e-be42-83dfbec33ab7', '215db66d-7a35-4780-b8a3-2bdbf2a28acc', '6cdf676d-266d-4be9-9ec1-319a9f282bb4', 'ca27c39d-2ebb-43f8-a567-7a76035edd88', '42948558-6dae-4000-8cdd-8e58821d85ee', 'f93e23d5-6192-412d-87d0-8c75221c509b', '2e1a4094-854c-494f-bcb5-e9d47447b2c2', 'e6b954be-9cd8-46e3-b76d-78d328a17fc2', '6fcbed5e-ed15-472e-b9c0-fb19262e16b6', '17e119d8-f3cb-4db9-a3ef-d498e3846fe8', '257afd9e-493e-4aac-91c7-f2565cfdb4ff', '6db2e395-f733-48e4-b993-d9eebe98c146', '1d25864d-fd3a-4a83-a2fe-93f9ced03b82', '8d48842e-44c1-41e6-80dd-387f7759e54a', 'f021ef5f-b047-4b51-92fe-5ed91802b684', 'af76107b-6fcd-42fb-b8e0-cbd34d9f12

By looking at this, we can check the id of the first node and the number of nodes the document has been divided into.

Next, let’s see how each node has been transformed into embeddings.


In [14]:
### YOUR CODE HERE ###

index.vector_store.data.embedding_dict[node_example_id]

[0.000312104937620461,
 -0.016931476071476936,
 -0.004101456608623266,
 -0.03391828387975693,
 0.005961982533335686,
 0.02765197865664959,
 -0.026075027883052826,
 -0.009800613857805729,
 -0.020790858194231987,
 -0.027707310393452644,
 0.01549285463988781,
 0.03353096544742584,
 -0.0052115474827587605,
 -0.008666315115988255,
 -7.964728138176724e-05,
 0.0196703914552927,
 0.02303178794682026,
 0.015022535808384418,
 0.00025482807541266084,
 -0.014828874729573727,
 -0.014234060421586037,
 -0.004502610769122839,
 0.009648451581597328,
 -0.0033786871936172247,
 0.00293257599696517,
 0.00028227773145772517,
 0.01989171840250492,
 -0.03895346447825432,
 0.0005632586544379592,
 -0.023903261870145798,
 0.019822554662823677,
 -0.01882658526301384,
 -0.0034253732301294804,
 -0.005415582563728094,
 -0.039091791957616806,
 -0.003449580864980817,
 -0.007725677918642759,
 -0.000714988331310451,
 0.009613869711756706,
 -0.01399198453873396,
 0.03723818436264992,
 0.018148772418498993,
 0.00493143079

As you can see here, each node is automatically converted into embeddings, becoming vectors when the index is created.

Next, let’s verify why the total number of nodes is as shown. When splitting a document into nodes, Llamaindex uses a class called SentenceSplitter. This class allows us to divide the text data into predefined lengths.

In [15]:
### YOUR CODE HERE ###

from llama_index.core.node_parser import SentenceSplitter

parser = SentenceSplitter(chunk_size=1024, chunk_overlap=200)
# you can change chunk_size, chunk_overlap

nodes = parser.get_nodes_from_documents(documents)

print(len(nodes))

22


Each node has an attribute called `text`, allowing us to see what text data each node contains. Additionally, you can see that the texts overlap by the value of the `chunk_overlap`.

When checking the length of each text data, we can see that they are composed of different numbers of characters. This is because the `SentenceSplitter` class divides `documents` not by the number of characters but by tokens, which are a type of sub-word.

In [20]:
### YOUR CODE HERE ###

print(nodes[0].text)
print("\n----------------------\n")
print(nodes[1].text)
print("\n----------------------\n")
print(len(nodes[0].text), len(nodes[1].text))

What I Worked On

February 2021

Before college the two main things I worked on, outside of school, were writing and programming. I didn't write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.

The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district's 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.

The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in the

We can use the default tokenizer used by Llamaindex’s `SentenceSplitter` to directly check how many tokens each text consists of.

By running the code below, we can bring in the tokenizer of `gpt-3.5-turbo`, which is used by default in `SentenceSplitter`, and perform tokenization to see the results and check how many tokens are generated.

When we look at the tokenization results, we will see that they differ from the `chunk_size` we set. This is because, when creating nodes, the amount of text data to be tokenized is determined by considering the metadata as well, in order to compose nodes of consistent size including the amount of metadata.

In [21]:
### YOUR CODE HERE ###

import tiktoken
tokenizer = tiktoken.encoding_for_model("gpt-3.5-turbo")

encoded_text = tokenizer.encode(nodes[0].text)
print(f"Encoded text: {encoded_text}")
print(f"Encoded text length: {len(encoded_text)}")

decoded_text = tokenizer.decode(encoded_text)
print(f"Decoded text: \n\n{decoded_text}")

Encoded text: [3923, 358, 5664, 291, 1952, 271, 33877, 220, 2366, 16, 271, 10438, 7926, 279, 1403, 1925, 2574, 358, 6575, 389, 11, 4994, 315, 2978, 11, 1051, 4477, 323, 15840, 13, 358, 3287, 956, 3350, 23691, 13, 358, 6267, 1148, 7314, 16483, 1051, 10171, 311, 3350, 1243, 11, 323, 4762, 2103, 527, 25, 2875, 7493, 13, 3092, 7493, 1051, 25629, 13, 2435, 1047, 20781, 904, 7234, 11, 1120, 5885, 449, 3831, 16024, 11, 902, 358, 35706, 1903, 1124, 5655, 382, 791, 1176, 7620, 358, 6818, 4477, 1051, 389, 279, 29022, 220, 6860, 16, 430, 1057, 2978, 9474, 1511, 369, 1148, 574, 1243, 2663, 330, 695, 8863, 1210, 1115, 574, 304, 220, 24, 339, 12239, 11, 779, 358, 574, 220, 1032, 477, 220, 975, 13, 578, 2978, 9474, 596, 220, 6860, 16, 7077, 311, 387, 304, 279, 31741, 315, 1057, 27144, 1579, 2978, 11, 323, 856, 4333, 8269, 2999, 4798, 323, 358, 2751, 8041, 311, 1005, 433, 13, 1102, 574, 1093, 264, 13726, 24537, 40148, 596, 1208, 404, 1523, 1070, 11, 449, 682, 1521, 20167, 31348, 12933, 2001, 14266, 11

You can see that the result here matches the number we saw earlier.  

If you want to change the default `chunk_size` or `chunk_overlap` when you make `index`, you can use `transformations` argument in `VectorStoreIndex.from_documents()`.


In [22]:
### YOUR CODE HERE ###

text_splitter = SentenceSplitter(chunk_size=200, chunk_overlap=50)

index = VectorStoreIndex.from_documents(documents=documents, transformations=[text_splitter])

In [23]:
### YOUR CODE HERE ###

node_id = index.index_struct.nodes_dict

for key, value in node_id.items():
    print(key, value)
    node_example_id = value
    break

print("The number of nodes: ", len(node_id.values()))

print(node_id.values())

fb419e6f-57f3-4754-bc05-f58bdb2e914e fb419e6f-57f3-4754-bc05-f58bdb2e914e
The number of nodes:  147
dict_values(['fb419e6f-57f3-4754-bc05-f58bdb2e914e', '42a5a724-c76b-46ac-8566-8c4444e2d824', '522de180-8e9d-45ba-b8b3-79f1e5d22db4', 'a18a7df1-97f4-49da-a973-d11b92666df9', 'b68c88d4-1db5-490a-a873-29d41f8ac320', 'db510737-e1f8-4977-a632-74550d0badc8', '0b7ad149-340c-41bd-ad01-bcb7848bc333', '7402217d-6b24-4c61-b7c4-556ab7ec6bc4', '3e1c1034-92bd-4588-944c-58309daacaa5', '7e95b223-3421-4e73-8d9c-4e2c4d62b740', '45f5c0c2-14ef-4c71-a403-42beb1848f0c', '40a395b7-2222-4f58-ab4a-83a0e42d85b6', 'da18c3cd-3d79-46b5-94c4-c4844833d45c', 'c4208d9d-ccf2-4757-897a-f8229a591577', '34b65271-4340-40ae-a669-9b439c6bc972', '8929a2fa-1795-4276-8607-8eb037b5378d', 'e2dd9a90-42f0-4743-ae4a-b270d1df717b', '169644cb-b99e-4dc3-b7e0-2e59677d39e4', '743456d4-86ca-4499-bd2f-948993f8c971', '9a48a6d1-27d1-4167-b05f-e0b5f195f697', 'bfb93252-c7a5-4ac4-a3cd-18d5919294df', '0f16785e-2744-48a4-b9cb-e56acec55804', '1435a3

But, in this practice, we will use default `SentenceSplitter`.

In [24]:
### YOUR CODE HERE ###

index = VectorStoreIndex.from_documents(documents)

Once the index is created, Llama Index provides a way to immediately create a query engine using the index. With the following code, we can create a query engine using the index.

In [25]:
### YOUR CODE HERE ###

query_engine = index.as_query_engine()

### 4. Conduct Activity

Now let's actually use the query engine. We will check the results of query engine using the following questions.

#### 4-1. Answerable Questions from Data

In this activity, we will check how query engine will answer **a question related to the given data, and the information needed for the correct answer is directly provided in the data**.

#### 4-2. Unrelated Question to the Data

In this activity, we will check how query engine will answer **a question that has absolutely nothing to do with the given data**.

#### 4-3. Question Requiring Complex Reasoning

In this activity, we will check how query engine will answer **a question that is hard to answer, which means that question is related to the given data, but the information needed for the correct answer is not directly given in the data**.

### 4-1. Answerable Questions from Data
First, let's ask the query engine some questions that it can answer and some that it cannot.

For example, we could ask what program the author of this data has written about. The answer to this question is in the second sentence of the txt file.

#### Question: **What is the first programs the author tried writing?**

#### In paul_graham_essay: **The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing."**

Intuitively, it is expected that the query engine will generate the correct answer, which is IBM 1401. Let's verify that.

In [27]:
### YOUR CODE HERE ###

text_chunk = "The first programs I tried writing were on the IBM 1401 that our school district used for what was then called \"data processing.\""

if text_chunk in documents[0].text:
    print("True")
else:
    print("False")

True


In [28]:
### YOUR CODE HERE ###

response = query_engine.query("What is the first programs the author tried writing?")

print(response)

The first programs the author tried writing were on the IBM 1401 using an early version of Fortran.


<img src="https://xe.obg.co.kr/files/attach/images/4199/352/004/7e2189cf45a27f9b9cda5fef28c1dd5f.gif">

As expected, the results are accurate.  


### 4-2. Unrelated Question to the Data

This time, We will check whether the query engine performs its intended function well.  
A query engine is **a generic interface that allows users to ask questions about external data**.  
Therefore, **it should not be able to answer questions that cannot be answered through external data**.

The reason is that if the query engine can answer questions that are completely unrelated to the external data, it implies that the query engine can freely use parameterized knowledge. This, in turn, means that the answers from the query engine may contain information that is incorrect or not up-to-date.

<img src="https://media0.giphy.com/media/ANbD1CCdA3iI8/200w.gif?cid=6c09b952ooe7fzryithl047ty5npdt1xd50tlu9gcpqnzh87&ep=v1_gifs_search&rid=200w.gif&ct=g" width="200" height="200"><img src="https://st2.depositphotos.com/4421345/11492/v/950/depositphotos_114921728-stock-illustration-public-speaking-robot.jpg" width="200">

So, let’s pass on a question that has nothing to do with the original data but that the query engine’s LLM can answer.  
For example, we could use a question like this.

#### Question: **How many countries participated in the production of the space station?**
#### Answer: **15**

In [29]:
### YOUR CODE HERE ###

response = query_engine.query("How many countries participated in the production of the space station?")

print(response)

There is no information provided in the context regarding the number of countries that participated in the production of the space station.


<img src = "https://img.danawa.com/images/descFiles/6/164/5163558_1_16652362092043267.gif
" width = 200>

It doesn't produce the correct answer as expected. However, we can find what kinds of LLM used in query engine through the official LlamaIndex documentation.

Actually, Llamaindex uses `gpt-3.5-turbo` as LLM of response synthesizer. So, we can ask `gpt-3.5-turbo` to answer the question.

Let's check this out with a simple code. We can use the following code to separately utilize the OpenAI LLM.  



In [30]:
### YOUR CODE HERE ###

def generate_answer(question):
    messages = [
        {
            "role": "user",
            "content": question,
        },
    ]
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        temperature=0,
        messages=messages,
    )

    return response.choices[0].message.content

In [31]:
### YOUR CODE HERE ###

question = "How many countries participated in the production of the space station?"

print(generate_answer(question))

The International Space Station (ISS) is a collaborative project involving five space agencies from 15 countries. These countries are the United States, Russia, Japan, Canada, and 11 European countries represented by the European Space Agency.


In fact, as in the example above, OpenAI LLM already knows the answer to this question.  
However, since the relevant information cannot be found in the txt file that query engine can refer to here, it does not generate a correct answer for a question that is already known.  

<img src="https://i.imgur.com/sxSRQks.png">

From this, we can understand that LLM in the response synthesizer of the query engine uses the information from the retrieved passages produced by the retriever, excluding its own parametrized knowledge.  

### 4-3. Question Requiring Complex Reasoning

Now, let’s explore more deeply. There is a question related to the data that is hard to answer. We will see how the query engine responds to such questions.

The question we will ask in this section is who is the author of the `paul_graham_essay.txt`. Becuase the original text file does not mention the author's name directly, this is a complex question.

Question: **Who is the author?**

Let's see whether the writer's name is actually not in the original text file or not.

In [32]:
### YOUR CODE HERE ###

wrapper = textwrap.TextWrapper(width=80)

with open(path_to_txt, 'r', encoding='utf-8') as file:
    for line in file:
        if 'paul graham' in line.lower():
            formatted_text = wrapper.fill(line.strip())
            print(formatted_text.split('.')[-1])


Occasionally after wrestling for hours with some gruesome bug I'd check Twitter
or HN and see someone asking "Does Paul Graham still code?"


This is the only sentence in the original txt file that mentions 'Paul Graham'.  
Now, let's read it ourselves and consider whether it actually specifies the author's name.

<br/>

Someone might infer that the author's name is Paul Graham from this sentence, but I think it's not certain

In my opinion, this sentence does not explicitly state the author's name, so it does not provide enough useful information

<br/>

However, opinions on this may vary, so let's verify it ourselves.


In [33]:
### YOUR CODE HERE ###

response_complex = query_engine.query("Who is the author?")

print(response_complex)

Paul Graham


<img src="https://i3.ruliweb.net/ori/21/07/30/17af799f631524cc2.gif">

Hmm... The query engine answers better than we think.  
Then, let's see what data were referred to by the query engine.
<br/>

<img src="https://i.imgur.com/sxSRQks.png">

The query engine made using `as_query_engine` uses a retriever generated directly from the index by using `as_retriever` method.   
Therefore, if we look at the results of this retriever, we can see which retrieved passages the query engine has seen.


In [34]:
### YOUR CODE HERE ###

retriever = index.as_retriever()

ret_passages = retriever.retrieve("Who is the author?")

for i in range(len(ret_passages)):
  print("###Retrieved Passage\n", ret_passages[i].text)
  print("\n\n\n")

###Retrieved Passage
 Large numbers of former students kept in touch with her, including me. After I moved to New York I became her de facto studio assistant.

She liked to paint on big, square canvases, 4 to 5 feet on a side. One day in late 1994 as I was stretching one of these monsters there was something on the radio about a famous fund manager. He wasn't that much older than me, and was super rich. The thought suddenly occurred to me: why don't I become rich? Then I'll be able to work on whatever I want.

Meanwhile I'd been hearing more and more about this new thing called the World Wide Web. Robert Morris showed it to me when I visited him in Cambridge, where he was now in grad school at Harvard. It seemed to me that the web would be a big deal. I'd seen what graphical user interfaces had done for the popularity of microcomputers. It seemed like the web would do the same for the internet.

If I wanted to get rich, here was the next train leaving the station. I was right about tha

This is quiet lone passages. So, it is hard to check whether there is any evidence of query engine's response or not.  
Can you find any supporting evidence of the question?  

Let's see if 'Paul Graham' is in there.



In [36]:
### YOUR CODE HERE ###

passages = ''

for i in range(len(ret_passages)):
  passages += ret_passages[i].text

if 'paul graham' in passages.lower():
    print("The word 'Paul Graham' is found in the passages.")
else:
    print("The word 'Paul Graham' is not found in the passages.")

The word 'Paul Graham' is not found in the passages.


<img src="https://i2.ruliweb.net/ori/21/07/24/17ad8e756ba54811a.gif" width="400">

???

Retriever couldn't find the only sentence with the word Paul Graham on it. Then how did query engine know that the answer was Paul Graham?

To verify this, let’s using the query engine’s LLM as we did in the previous section and ask the same question. We will use a prompt as similar as possible to the one used in LlamaIndex response synthesizer, but with an additional instruction to provide the reason for the LLM’s answer.  

The prompt used by the original Response Synthesizer is as follows.


<img src = "https://i.imgur.com/ZXUDAJi.jpeg" width="450">  


<br/>

Therefore, we will write the code as shown below to check the results of the LLM.



In [37]:
### YOUR CODE HERE ###

ret_context = ""
for ret_result in ret_passages:
  ret_context += ret_result.text

question = f"""Context information is below.
---------------------
{ret_context}
---------------------
Given the context information and not prior knowledge,
answer the query and the reasons of answer.
Query: Who is the author?
Answer:
"""

print(generate_answer(question))

The author of the text is Paul Graham. This can be inferred from the personal experiences and details shared in the text, such as working on programming projects, starting a company called Viaweb, and studying philosophy and AI in college, all of which align with Paul Graham's background and experiences.


Well... it seems like query engine can **capture useful information not only in direct evidence but also in indirect content by using its prior knowledge, i.e., parametrized knowledge**.  

Therefore, we can consider that the query engine has the ability to provide correct answers even when the given query is not explicitly answered in the data.  

However, the problem is that **without such an explanation, the reasoning process cannot be trusted** because it is impossible to know whether the LLM used reliable information in its reasoning.

<br/>

Here’s what we can infer from this:

the query engine has the capability to answer not only questions directly addressed in the data **but also questions that can be indirectly inferred from the data.**  
However, in such cases, **the reliability of the answers is compromised because we cannot verify the information used and the reasoning steps.**   

To address this issue, we can use one of the following two methods.  


#### **1.** Force the response synthesizer to utilize only the information in a given retrieved passages.
#### **2.** When the response synthesizer generates a summary, make sure that the reason is also generated.

## II. Use Database Management Method

Now, let’s consider a scenario. If we have new data and want to create a query engine using it, we could recreate the `document`, `index`, and `query engine` as we did above.

However, this is quite cumbersome and time-consuming. In addition, if you want to insert new data so that query engine uses that data, the above method is very expansive because a new index must be generated for every inserts operation.

So, Llama Index is providing the following functions to manage data smoothly.

### 1. Insert

We can insert new data to the existing query engine.

To check this, we will ask a question which can't get useful information from the data before inserting new data, and see what happens after inserting data.

In this section, the following data and question will be used.

Inserted Data: **"Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 900–1,400 °C and at pressures of 5–6 GPa."**  
Question: **What are the temperature and air pressure conditions under which natural diamonds are produced?**


In [38]:
### YOUR CODE HERE ###

data_text = "Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 900–1,400 °C and at pressures of 5–6 GPa."
question = "What are the temperature and air pressure conditions under which natural diamonds are produced?"

res = query_engine.query(question)

print(res)

Natural diamonds are typically formed under high temperature and pressure conditions deep within the Earth's mantle.


<img src = "https://i.pinimg.com/originals/15/8b/ed/158bed9819e4fccf7e18a5eeeaf79c6b.png" width = 200>

Hmm… the query engine answers quite well. It seems the response synthesizer is using its parameterized knowledge. However, there are no specific details like 900–1,400°C and pressures of 5–6 GPa.

Anyway, it's time to insert the data.



In [39]:
### YOUR CODE HERE ###

docu = Document(text=data_text, id_="new_doc_id")

index.insert(docu)

new_query_engine = index.as_query_engine()

In [40]:
### YOUR CODE HERE ###

new_res = new_query_engine.query(question)

print(new_res)

Natural diamonds are formed in metallic melts at temperatures ranging from 900 to 1,400 degrees Celsius and at pressures of 5 to 6 gigapascals.


It's as expected. Query engine is generating an appropriate answer with detail using information in new `Document` object.

Then, what happens if we change the information in that data?

### 2. Update

If a Document is already present within an index, you can "update" a `Document` with the same doc `id_`. The `update_ref_doc` method receives a single `Document` object and updates the value of data that has the same `id`.

Or you can "refresh" all document at once. The `refresh_ref_doc` method finds the `id` of the `Document` object that came into the input, updates the data with the same `id`, or inserts the data if there is no other data with the same `id`

**`update_ref_doc`**  
**Input**: single `Document` object  
**Output**: None

**`refresh_ref_doc`**  
**Inpu**t: list of `Document` object  
**Output**: a boolean list, indicating which documents in the input have been refreshed in the index.

You will see `False` in that boolean list if text of document doesn't change.  

In both methods, we can set `delete_from_docstore` to `True` or `False`. A detailed description of this will be given in Delete.


<br/>

Let's check by changing the base question a little bit.  

I changed the information about temperature and air pressure in the question, so check it out for yourself.  



In [41]:
### YOUR CODE HERE ###

docu.set_content(value="Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 2,000–6,000 °C and at pressures of 8–9 GPa.")

output = index.update_ref_doc(
    docu,
    update_kwargs={"delete_kwargs": {"delete_from_docstore": True}},
)

print(output)

query_engine_update = index.as_query_engine()

res_update = query_engine_update.query(question)

print(res_update)

None
Natural diamonds are formed in metallic melts at temperatures ranging from 2,000 to 6,000 degrees Celsius and at pressures between 8 to 9 gigapascals.


In [42]:
### YOUR CODE HERE ###

docu.set_content(value="Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 6,000–8,000 °C and at pressures of 12–15 GPa.")

output = index.refresh_ref_docs(
    [docu]
)

print(output)

query_engine_refresh = index.as_query_engine()

res_refresh = query_engine_refresh.query(question)

print(res_refresh)

[True]
Natural diamonds are formed in metallic melts at temperatures of 6,000–8,000 °C and at pressures of 12–15 GPa.


In [44]:
docu

Document(id_='new_doc_id', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 6,000–8,000\u2009°C and at pressures of 12–15\u2009GPa.', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}')

You can check list of doc_id in the index. So, if you want more experience with other data, you can check whether your data is inserted properly by looking at that list

In [43]:
### YOUR CODE HERE ###

print(index.ref_doc_info.keys())

dict_keys(['3e0e10b3-e674-462e-a980-13b84f7f4546', 'new_doc_id'])


In [46]:
index.ref_doc_info

{'3e0e10b3-e674-462e-a980-13b84f7f4546': RefDocInfo(node_ids=['17f97f01-56c1-4206-88c3-efe6f338d9b1', '03d7d79d-115c-4f1c-97a6-68622a09da8b', '40ee4825-b4b4-4258-9e3f-0347e8f8b69d', '53863c6f-dc7b-4bd9-a4e7-be8443dd6c89', '5c16a7fd-8f1d-493b-b748-4073da9bf0dd', '3e037da6-8e97-49d6-be72-b6b58a4d8cb1', '2bc217d2-3ef6-45b1-87d6-f205b933521d', '9cb7ea8d-bffe-445a-aafb-bae58aec11c3', '012597bb-e92e-44e2-aad2-6c51257f145e', '94e52d81-0a9c-4143-8093-2ca20b15b330', '74cb1eb9-2c9f-4982-82fa-0ec4c9ae9414', '0d7b43b6-b1f2-4ac1-8be3-be6f0556c979', '9893a578-4e9d-46fe-a145-1b4992d60098', 'd8dcf580-4795-423e-8455-913836cbe247', 'b4c8d9d1-4647-4450-aaaf-bb78a68e3de0', 'ba77dfec-070c-495e-96a8-cb65eca0c7a2', '44b46e36-886a-46ed-b681-16b59878fcb9', 'd456d578-cc74-4a0e-bcc3-2c992cbcba26', '545ac543-6e44-43ca-99a3-b3e7b3c82e75', '20b8085d-fcf0-46ec-818a-d31cf22d5a1e', '9dffdb87-8e1b-4e81-b9c3-69de9d88a6ba', '04f794db-a560-4995-b216-d30198877e7a'], metadata={'file_path': 'C:\\ai-specialist\\RAG\\실습자료\\1일차

### 3. Delete

We wrapped the new information above with a `Document` object and inserted it into the index.

At this time, we were able to specify the `Document` object using `doc_id`. What this doc_id is can be confirmed as follows.

In [47]:
### YOUR CODE HERE ###

id = docu.doc_id

print(id)

new_doc_id


Using this, we can change the contents of data with the `doc_id` using the `delete` method of the index.

Let's delete the data related to a diamond, and see how query engine answers for the same question.

Through the value of `delete_from_docstroe`, it is possible to determine whether the data with that id actually disappears on the database, or disappears only on the index and remains in the database. Even if it disappears only on the index, the information of the data cannot be used by the query engine.



In [48]:
### YOUR CODE HERE ###

index.delete_ref_doc(id, delete_from_docstore=True)

query_engine_delete = index.as_query_engine()

res_delete = query_engine_delete.query(question)

print(res_delete)

Natural diamonds are typically formed under high temperature and pressure conditions deep within the Earth's mantle.


If the answer still contains very details of condition of natural diamonds, check the index whether that data was deleted properly.

The result of `index.ref_doc_info.keys()` has to contain only one doc_id.

In [49]:
### YOUR CODE HERE ###

print(index.ref_doc_info.keys())

dict_keys(['3e0e10b3-e674-462e-a980-13b84f7f4546'])


## III. Make a Custom Query Engine

In this section, we will define the existing query engine in more detail and modify its internal components to create the query engine we desire.
<br/>

The query engine can vary in its usage depending on the task requested after passing the retrieved documents to the LLM in response synthesizer. For instance, it can be tasked with summarizing the documents or directly answering the query.

Since the query engine can read through extremely long documents that exceed the LLM’s context limit, it can have a significant impact on RAG, depending on how it is used. Therefore, choosing which query engine to use is an important design choice when designing RAG.

This section is composed of the following four stages:
 <br/>
#### 1. Defining the query engine in detail
#### 2. Creating a custom query engine


### 1. Defining the query engine in detail

In this section, we will define the retriever and response synthesizer, which are internal components of the query engine, and use them to define a class that performs the role of the query engine.

In [50]:
### YOUR CODE HERE ###

class StandardQueryEngine(CustomQueryEngine):
    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)
        response_obj = self.response_synthesizer.synthesize(query_str, nodes)
        return response_obj

Initialize the retriever and response synthesizer, and then verify that the class functions correctly.


In [51]:
### YOUR CODE HERE ###

retriever = index.as_retriever()
synthesizer = get_response_synthesizer(response_mode="compact")
query_engine = StandardQueryEngine(
    retriever=retriever, response_synthesizer=synthesizer
)

response = query_engine.query("What did the author do growing up?")
print(str(response))

The author worked on writing and programming before college.


### 2. Creating a custom query engine

Now, let’s modify the query engine so that we can pass the desired task to the response synthesizer through a prompt.  

While there are many aspects of the retriever that can also be adjusted, in this exercise, we will focus on changing the content passed to the response synthesizer to experience more noticeable changes

from llama_index.core import PromptTemplate


In [52]:
### YOUR CODE HERE ###

simple_qa_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)

short_sum_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "give the summary of context, less than 20 words.\n"
    "Query: {query_str}\n"
    "Summary: "
)

In [53]:
### YOUR CODE HERE ###

from llama_index.llms.openai import OpenAI

class OurCustomQueryEngine(CustomQueryEngine):

    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer
    llm: OpenAI
    qa_prompt: PromptTemplate = simple_qa_prompt

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)

        context_str = "\n\n".join([n.node.get_content() for n in nodes])
        response = self.llm.complete(
            self.qa_prompt.format(context_str=context_str, query_str=query_str)
        )

        return str(response)

Now, let’s check if it truly works as we intended.


In [54]:
### YOUR CODE HERE ###

llm = OpenAI(model="gpt-3.5-turbo")

query_engine = OurCustomQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
    llm=llm,
    qa_prompt=simple_qa_prompt,
)

response = query_engine.query("What did the author do growing up?")
print(str(response))

The author worked on writing short stories and programming, starting with the IBM 1401 in 9th grade and later transitioning to microcomputers like the TRS-80. They wrote simple games, a rocket prediction program, and a word processor. They also applied to art schools and eventually attended RISD and the Accademia di Belli Arti in Florence.
